# Train SciBERT contrastive variants (Colab T4)
Trains the original 4 variants + 4 new BM25 hard-neg variants from Phase 1.2.
Total ~70-90 min on T4. Each variant is independent — you can rerun any single line.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
rm -rf /content/repo
git clone https://github.com/wswaileh/palestinian-drugs-data.git /content/repo
cd /content/repo && git checkout main
pip install -q -r requirements.txt
pip uninstall -q -y wandb codecarbon mlflow comet-ml || true

In [ ]:
%cd /content/repo
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## Original 4 variants (already trained — skip if checkpoints already on Drive)
If `/content/drive/MyDrive/scibert_ft/{a_only,b_only,ab_random,ab_atc}/model.safetensors` already exist, you can skip this cell.

In [ ]:
%%bash
set -e
export WANDB_DISABLED=true
export WANDB_MODE=disabled
for variant in a_only b_only ab_random ab_atc; do
  if [ -f /content/drive/MyDrive/scibert_ft/$variant/model.safetensors ]; then
    echo "==== skipping $variant (already trained) ===="
    continue
  fi
  echo "==== training variant: $variant ===="
  python -m src.training.train --variant $variant \
    --output-dir /content/drive/MyDrive/scibert_ft
done

## NEW: BM25 hard-negative variants (Phase 1.2)
ab_bm25 / ab_atc_bm25 train on combined A+B; b_only_bm25 / b_only_atc_bm25 train on Type B only.
The `_atc_bm25` variants emit 2× triples (one per neg type) — slower but more signal.

In [ ]:
%%bash
set -e
export WANDB_DISABLED=true
export WANDB_MODE=disabled
for variant in ab_bm25 ab_atc_bm25 b_only_bm25 b_only_atc_bm25; do
  echo "==== training variant: $variant ===="
  python -m src.training.train --variant $variant \
    --output-dir /content/drive/MyDrive/scibert_ft
done

In [ ]:
%%bash
ls -la /content/drive/MyDrive/scibert_ft/